# 🏌️ Mini Caddie — Custom Golf Model Training
Train a YOLOv8n model on the unified golf dataset (ball, club, swing)
Export to ONNX for Hailo compilation

**Resume support built in** — if Colab disconnects, just re-run from Step 1 and it picks up where it left off.

## Step 1: Mount Google Drive

In [ ]:
# STEP 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Install YOLOv8 + Verify GPU
This installs ultralytics and confirms you're on a T4 GPU (not CPU!).

In [ ]:
# STEP 2: Install YOLOv8
!pip install ultralytics -q

# Verify GPU is available — if this says CPU, fix runtime type!
import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ NO GPU! Go to Runtime → Change runtime type → T4 GPU')

## Step 3: Unzip Dataset
Extracts the dataset from Google Drive. Skips if already extracted.

In [ ]:
# STEP 3: Unzip dataset (skips if already extracted)
import os, zipfile

zip_path = '/content/drive/MyDrive/unified-golf-dataset.zip'
extract_path = '/content/dataset'

if os.path.exists(os.path.join(extract_path, 'unified', 'data.yaml')):
    print('✅ Dataset already extracted — skipping!')
else:
    os.makedirs(extract_path, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_path)
    print('✅ Dataset extracted!')

!ls /content/dataset/unified/
!cat /content/dataset/unified/data.yaml

## Step 4: Fix data.yaml Paths

In [ ]:
# STEP 4: Fix data.yaml paths for Colab
yaml_path = '/content/dataset/unified/data.yaml'

with open(yaml_path, 'r') as f:
    c = f.read()

c = c.replace('../train/images', '/content/dataset/unified/train/images')
c = c.replace('../valid/images', '/content/dataset/unified/valid/images')
c = c.replace('../test/images', '/content/dataset/unified/test/images')

with open(yaml_path, 'w') as f:
    f.write(c)

print('✅ data.yaml paths updated!')
print(c)

## Step 5: Train YOLOv8 (with auto-resume)
This cell automatically detects if a previous training run exists and resumes from the last checkpoint.

- **First run:** Starts fresh from `yolov8n.pt`, trains 50 epochs
- **After disconnect:** Re-run this cell — it finds `last.pt` and resumes automatically

Checkpoints are also copied to Google Drive every `save_period` epochs so they survive disconnects.

In [ ]:
# STEP 5: Train YOLOv8 with auto-resume
# Automatically resumes from last checkpoint if one exists
import os, shutil
from ultralytics import YOLO

run_dir = '/content/runs/detect/mini_caddie_golf'
last_ckpt = os.path.join(run_dir, 'weights', 'last.pt')
drive_ckpt = '/content/drive/MyDrive/mini_caddie_checkpoints/last.pt'

if os.path.exists(last_ckpt):
    print('📎 Found local checkpoint — resuming training...')
    model = YOLO(last_ckpt)
    results = model.train(resume=True)
elif os.path.exists(drive_ckpt):
    print('📎 Found Drive checkpoint — restoring and resuming...')
    os.makedirs(os.path.join(run_dir, 'weights'), exist_ok=True)
    shutil.copy(drive_ckpt, last_ckpt)
    model = YOLO(last_ckpt)
    results = model.train(resume=True)
else:
    print('🚀 No checkpoint found — starting fresh training...')
    model = YOLO('yolov8n.pt')
    results = model.train(
        data='/content/dataset/unified/data.yaml',
        epochs=50,
        imgsz=640,
        batch=16,
        name='mini_caddie_golf',
        patience=20,
        save=True,
        save_period=5,  # Save checkpoint every 5 epochs (was 10)
    )

# Backup final weights to Drive
drive_backup_dir = '/content/drive/MyDrive/mini_caddie_checkpoints'
os.makedirs(drive_backup_dir, exist_ok=True)
if os.path.exists(os.path.join(run_dir, 'weights', 'last.pt')):
    shutil.copy(
        os.path.join(run_dir, 'weights', 'last.pt'),
        os.path.join(drive_backup_dir, 'last.pt')
    )
    print('✅ Checkpoint backed up to Google Drive!')

## Step 5b: Backup Checkpoints to Drive (run after disconnect!)
If training gets interrupted, run this cell to copy the latest checkpoint to Google Drive before the runtime recycles.

**Do this immediately if you see training stop unexpectedly!**

In [ ]:
# STEP 5b: Emergency checkpoint backup to Drive
# Run this if training gets interrupted to save your progress
import os, shutil

run_dir = '/content/runs/detect/mini_caddie_golf'
drive_backup_dir = '/content/drive/MyDrive/mini_caddie_checkpoints'
os.makedirs(drive_backup_dir, exist_ok=True)

copied = False
for fname in ['last.pt', 'best.pt']:
    src = os.path.join(run_dir, 'weights', fname)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(drive_backup_dir, fname))
        print(f'✅ {fname} backed up to Drive!')
        copied = True

if not copied:
    print('⚠️ No weights found yet — training may not have reached a save point.')
else:
    print('📁 Checkpoints safe in Google Drive. You can resume later with Step 5.')

## Step 6: Export to ONNX for Hailo

In [ ]:
# STEP 6: Export best model to ONNX
from ultralytics import YOLO

best = YOLO('/content/runs/detect/mini_caddie_golf/weights/best.pt')
best.export(format='onnx', opset=11)
print('✅ ONNX exported!')

## Step 7: Save Models to Google Drive

In [ ]:
# STEP 7: Save final models to Google Drive
import shutil

shutil.copy('/content/runs/detect/mini_caddie_golf/weights/best.onnx',
            '/content/drive/MyDrive/mini_caddie_golf_best.onnx')
shutil.copy('/content/runs/detect/mini_caddie_golf/weights/best.pt',
            '/content/drive/MyDrive/mini_caddie_golf_best.pt')

print('✅ Models saved to Drive!')
print('  - mini_caddie_golf_best.onnx (for Hailo compilation)')
print('  - mini_caddie_golf_best.pt (PyTorch backup)')

## Step 8: Quick Test on Validation Images

In [ ]:
# STEP 8: Test on validation images
results = best.predict(
    source='/content/dataset/unified/valid/images',
    save=True,
    conf=0.25,
    max_det=10,
    project='/content/runs/detect',
    name='mini_caddie_test'
)
print('✅ Test predictions saved!')
print('Check /content/runs/detect/mini_caddie_test/ for results')